# Which "Hits" Tree Is In My File?

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

GATE does not write one "Hits" tree. What it holds depends on the simulation:
a detector placed in a system adds a branch per level of the system, the
Compton camera output replaces a whole block of branches with another, and the
`GateToTree` module writes the same data in another order under another name.

This notebook takes three real files with three different structures, asks the
package which structure each one holds, and looks at what tells them apart. It
then turns to what the package does with that knowledge: it checks every tree
it reads against the structure it recognised, and the check is deliberately
stricter about a missing branch than about one too many.

## What will you learn from this notebook

- how to ask which structure a "Hits" tree has, without reading its data
- how the structures differ: what a system adds, and what the Compton camera
  output swaps
- how to list the branches a structure holds, with their types
- how to recognise and check a list of branch names that did not come from a
  file
- why a missing branch is an error and an extra one only a warning
- what reading with the check turned off saves, and what it costs

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The one place this notebook may write, and the three files it reads. Each one
is the "Hits" tree of a GATE simulation cut to its first 500 entries; what they
hold is described in [`../data/README.md`](../data/README.md).

In [2]:
output = paths.output_dir("tutorial-05-hits-tree-variants")

files = {
    "A1": paths.input_example("hits-a1-no-system.root"),
    "A2": paths.input_example("hits-a2-cylindrical-pet.root"),
    "A4": paths.input_example("hits-a4-compton-camera.root"),
}

In [3]:
from opengate_gate_tree import (
    GateSystemType,
    GateTree,
    GateTreeError,
    HitsTreeVariant,
    OutputFileFormat,
    RootFile,
    describe_hits_tree,
    detect_hits_variant,
    expected_branches,
    read_tree,
    summarise_hits_tree,
    supported_variants,
    validate_hits_tree,
    variant_reference,
    write_tree,
)

### The Structures The Package Knows

Six of them. Each has a label, taken from the reference material the schemas
were written from, so that a file can be talked about without describing it
every time — the labels turn up in reports and in error messages:

In [4]:
for variant in supported_variants():
    print(f"{variant_reference(variant)}  {variant.value}")

A1  No system
A2  System
A3  System with septal penetration
A4  No system, Compton camera output
A5  System, Compton camera output
B1  GateToTree common output


### Asking A File

`detect_hits_tree()` looks at the branch names of the tree and nothing else:
no entry is read, so the answer costs the same for a file of five hundred hits
as for one of fifty million. `summarise_hits_tree` puts the answer on one line:

In [5]:
detections = {}
for label, path in files.items():
    with RootFile(path) as root_file:
        detections[label] = root_file.detect_hits_tree()
    print(summarise_hits_tree(detections[label]))

Hits tree variant: No system (A1), stored as 'Hits', 40 branches
Hits tree variant: System (A2), stored as 'Hits', system cylindricalPET / OPET, 46 branches
Hits tree variant: No system, Compton camera output (A4), stored as 'Hits', 30 branches


Three files, three structures. The answer itself is a `HitsTreeDetection`,
whose fields can be used in code — which structure, which naming scheme of the
system, how many identifier branches, under which name the tree is stored:

In [6]:
detections["A2"]

HitsTreeDetection(variant=<HitsTreeVariant.SYSTEM: 'System'>, system=<GateSystemType.CYLINDRICAL_PET: 'cylindricalPET'>, system_depth=6, tree_name='Hits', branch_count=46)

### What A System Adds

The structures are compared on the branches they describe, which
`expected_branches` returns in the order GATE writes them. A structure with a
system needs the naming scheme of that system to be complete, since the names
of the identifier branches depend on it:

In [7]:
def names(variant, system=None):
    return [spec.name for spec in expected_branches(variant, system)]


a1 = names(HitsTreeVariant.NO_SYSTEM)
a2 = names(HitsTreeVariant.SYSTEM, GateSystemType.CYLINDRICAL_PET)

len(a1), len(a2), [name for name in a2 if name not in a1]

(40,
 46,
 ['gantryID', 'rsectorID', 'moduleID', 'submoduleID', 'crystalID', 'layerID'])

A2 is A1 with six identifier branches, one per level of the `cylindricalPET`
hierarchy, from the gantry down to the layer. Nothing else differs. Each
branch comes with its kind and the type GATE stores it with:

In [8]:
expected_branches(HitsTreeVariant.SYSTEM, GateSystemType.CYLINDRICAL_PET)[22]

BranchSpec(name='gantryID', kind=<BranchKind.INTEGER: 'integer'>, dtype='int32')

What the identifiers hold, in the file. `volumeID` already carries the whole
path through the geometry, ten levels of it; the six identifier branches repeat
a part of that path, one level each:

In [9]:
identifiers = ["gantryID", "rsectorID", "moduleID", "submoduleID", "crystalID", "layerID"]

a2_data = read_tree(files["A2"], GateTree.HITS, [*identifiers, "volumeID"])
a2_frame = a2_data.to_dataframe()
a2_frame[identifiers].nunique()

gantryID         1
rsectorID      121
moduleID         8
submoduleID      8
crystalID       25
layerID          1
dtype: int64

In [10]:
all(
    (a2_frame[name] == a2_data["volumeID"][:, level + 1]).all()
    for level, name in enumerate(identifiers)
)

True

In this file the six identifiers are `volumeID[1]` to `volumeID[6]`, level by
level: one gantry, 121 of the rotating sectors hit, 8 modules, 8 submodules, 25
crystals and one layer. The identifier branches are the same numbers under
names that say which level they belong to.

### What The Compton Camera Output Swaps

A4 is the Compton camera output of a simulation without a system. It keeps the
first 22 branches of A1, up to `processName`, and replaces everything after
them:

In [11]:
a4 = names(HitsTreeVariant.NO_SYSTEM_CC)

print("A1 only:", [name for name in a1 if name not in a4])
print("A4 only:", [name for name in a4 if name not in a1])

A1 only: ['momDirX', 'momDirY', 'momDirZ', 'photonID', 'nPhantomCompton', 'nCrystalCompton', 'nPhantomRayleigh', 'nCrystalRayleigh', 'nInteractions', 'primaryID', 'axialPos', 'rotationAngle', 'comptVolName', 'RayleighVolName', 'sourceType', 'decayType', 'gammaType', 'decayIndex']
A4 only: ['sourceEnergy', 'sourcePDG', 'nCrystalConv', 'nCrystalCompt', 'nCrystalRayl', 'energyFinal', 'energyIniT', 'postStepProcess']


The direction of flight, the interaction counters and the branches a
`PositroniumSource` writes are gone; the energy of the source, the energy of
the particle before and after the step, and the process that ended the step
take their place. An analysis written for A1 will not find `momDirX` here, and
the time to learn that is before the analysis runs — which is what recognising
the structure is for.

`describe_hits_tree` lists the branches of a recognised tree with their types.
Given only the detection, it describes the schema and needs no open file:

In [12]:
print(describe_hits_tree(detections["A4"]))

Detected Hits tree variant: No system, Compton camera output (A4)
Tree name in the file: Hits
Branches (30):
  - PDGEncoding / int32
  - trackID / int32
  - parentID / int32
  - trackLocalTime / float64
  - time / float64
  - edep / float32
  - stepLength / float32
  - trackLength / float32
  - posX / float32
  - posY / float32
  - posZ / float32
  - localPosX / float32
  - localPosY / float32
  - localPosZ / float32
  - sourcePosX / float32
  - sourcePosY / float32
  - sourcePosZ / float32
  - sourceID / int32
  - eventID / int32
  - runID / int32
  - volumeID / int32[10]
  - processName / text
  - sourceEnergy / float32
  - sourcePDG / int32
  - nCrystalConv / int32
  - nCrystalCompt / int32
  - nCrystalRayl / int32
  - energyFinal / float32
  - energyIniT / float32
  - postStepProcess / text


### Recognising A List Of Names

Recognition works on branch names, so it does not need a file at all. That is
useful when the names come from somewhere else — a tree written by other code,
or a list in a configuration:

In [13]:
detect_hits_variant(a2)

HitsTreeDetection(variant=<HitsTreeVariant.SYSTEM: 'System'>, system=<GateSystemType.CYLINDRICAL_PET: 'cylindricalPET'>, system_depth=6, tree_name=None, branch_count=46)

Recognition looks at a few branches that say what the simulation did, rather
than at the whole list. Seven names are enough to be recognised as a system
structure with the `SPECThead` naming scheme:

In [14]:
detect_hits_variant(
    ["volumeID", "headID", "crystalID", "pixelID", "unused3ID", "unused4ID", "unused5ID"]
)

HitsTreeDetection(variant=<HitsTreeVariant.SYSTEM: 'System'>, system=<GateSystemType.SPECT_HEAD: 'SPECThead'>, system_depth=6, tree_name=None, branch_count=7)

That is deliberate. A file from a GATE build that adds or drops a branch is
still recognised as the structure it is, so the check that follows can say
*which* branch is missing, instead of answering "unknown" to everything that is
not an exact match. Only a list holding none of the branches that identify a
structure is refused outright:

In [15]:
try:
    detect_hits_variant(["eventID", "edep"], tree_name="Hits")
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

UnknownHitsVariantError: Tree 'Hits' does not hold a branch that identifies a known Hits structure: neither 'volumeID' nor 'volumeID[0]' is present. Branches present: ['eventID', 'edep'].


### Checking A Tree Against Its Structure

`validate_hits_tree` takes the branch names and the type each one is stored
with, and checks them against the structure they are recognised as. Reading a
"Hits" tree runs the same check before any data is loaded. Here it runs on the
A1 schema itself, changed one way at a time:

In [16]:
a1_types = {spec.name: spec.dtype for spec in expected_branches(HitsTreeVariant.NO_SYSTEM)}

validate_hits_tree(a1, a1_types)

HitsTreeDetection(variant=<HitsTreeVariant.NO_SYSTEM: 'No system'>, system=None, system_depth=None, tree_name=None, branch_count=40)

A branch missing from the tree is an error, and so is a branch stored with
another type:

In [17]:
without_edep = [name for name in a1 if name != "edep"]

try:
    validate_hits_tree(without_edep, a1_types)
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

HitsTreeValidationError: The tree was recognised as the 'No system' structure (A1), but does not match it. Branches missing from the tree: ['edep'].


In [18]:
try:
    validate_hits_tree(a1, {**a1_types, "edep": "float64"})
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

HitsTreeValidationError: The tree was recognised as the 'No system' structure (A1), but does not match it. Branches stored with another type: 'edep' expected float32, stored as float64.


A branch beyond the structure is not an error. The package says so in its log,
which stays silent until the application gives it somewhere to write — here, a
handler printing the level and the message:

In [19]:
import logging
import sys

from opengate_gate_tree import LOGGER_NAME

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logging.getLogger(LOGGER_NAME).addHandler(handler)

In [20]:
validate_hits_tree([*a1, "myCounter"], {**a1_types, "myCounter": "int32"})

HitsTreeDetection(variant=<HitsTreeVariant.NO_SYSTEM: 'No system'>, system=None, system_depth=None, tree_name=None, branch_count=41)

The tree is recognised and accepted, with one warning naming the branch the
structure does not describe.

The asymmetry is the point. The reference files come from a GATE build carrying
patches, and adding a branch is an ordinary thing for a build to do: refusing
such a file would turn the package away from the simulations it exists for. A
missing branch is different — it means the data is not what it was taken for,
and an analysis that relies on that branch would fail later, or worse, answer
with something else.

### Turning The Check Off

A file the package cannot recognise, or one it recognises and rejects, can
still be read: `validate=False` in the library, `--skip-hits-validation` on
the command line. To see what that costs, write the A1 tree without `edep` —
a selection on the `TreeData` keeps `volumeID` as an array, so the file is
still recognised as A1:

In [21]:
a1_data = read_tree(files["A1"], GateTree.HITS)
broken = a1_data.select([name for name in a1_data.branch_names if name != "edep"])

broken_file = write_tree(broken, output / "a1-without-edep.root", OutputFileFormat.ROOT)
broken_file.name

'a1-without-edep.root'

Read with the check, the file is refused before a single entry is loaded, and
the message says what is wrong with it:

In [22]:
try:
    read_tree(broken_file, GateTree.HITS)
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

HitsTreeValidationError: Tree 'Hits' was recognised as the 'No system' structure (A1), but does not match it. Branches missing from the tree: ['edep'].


Read without it, the file opens as if nothing were wrong:

In [23]:
unchecked = read_tree(broken_file, GateTree.HITS, validate=False)
unchecked

TreeData(tree='Hits', entries=500, branches=39)

And that is the price. Nothing says the deposited energy is missing until the
analysis reaches for it, by which point the question is no longer "is this the
file I think it is" but a `KeyError` in the middle of something else:

In [24]:
try:
    unchecked.to_dataframe()["edep"]
except KeyError as error:
    print(f"KeyError: {error}")

KeyError: 'edep'


Turning the check off is right for a file that is a selection on purpose — an
output of this package read back, for instance — and for a structure the
package does not know yet. For a file straight out of GATE it hides exactly the
mistake it exists to catch.

## Summary

The package tells the structures of a "Hits" tree apart by a few branches that
say what the simulation did, lists the branches each structure holds, and
checks every tree it reads against the structure it recognised: a missing
branch or a wrong type is an error, an extra branch a warning. The check can be
turned off, and what that costs is finding out about a missing branch only when
the analysis needs it.

## Bibliography

- [The "Hits" Tree](https://opengate-gate-tree.readthedocs.io/en/latest/guide/hits.html)
  — every structure, how it is recognised, what validation checks, and the
  branch lists
- [Loading files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/loading.html)
  — what reading checks before any data is loaded
- [Exporting files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/export.html)
  — reading your own output back with the check turned off
- [Command line](https://opengate-gate-tree.readthedocs.io/en/latest/guide/cli.html)
  — `--skip-hits-validation`

## Changelog

- v1: written.